In [ ]:
print("check 123 ")

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def pdf_date_to_ist(pdf_date):
    dt = datetime.strptime(
        pdf_date[2:16],
        "%Y%m%d%H%M%S"
    )
    dt = dt.replace(tzinfo=ZoneInfo("UTC"))
    return dt.astimezone(
        ZoneInfo("Asia/Kolkata")
    ).strftime("%Y-%m-%d %H:%M:%S IST")

print(
    pdf_date_to_ist(
        "D:20260607071500Z00'00'"
    )
)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter



pdf_path = "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023.pdf"
try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
    )

    print("Total Pages:", len(documents))
    for i in range(5):
        print(f"------------New Page {i}________________\n\n")
        # print(documents[i].page_content)
        
        chunks = splitter.split_documents(documents[i])
        print(chunks)

except Exception as e:
    print(e)


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="Hello World",
        metadata={"page": 1}
    )
]

chunks = splitter.split_documents(docs)
chunks

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

pdf_path = "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Pages:", len(documents))
print("Chunks:", len(chunks))

print(chunks[0])



for i in range(5):
    print(f"\nChunk {i+2}")
    idx= i+2
    if (idx & 1):
        chunks[idx].metadata["law"] = "BNSS 2023"
        chunks[idx].metadata["language"] = "English"
    # print(chunks[i+2].metadata)
    str= chunks[idx].metadata["creationdate"]
    source= "THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023"
    chunks[idx].metadata["source"]= source
    # chunks[idx].metadata["creationdate"]= pdf_date_to_ist(str)
    # chunks[idx].metadata["moddate"]= pdf_date_to_ist(chunks[idx].metadata["moddate"])
    print(json.dumps(chunks[idx].metadata, indent=4))




Pages: 249
Chunks: 1210
page_content='vlk/kkj.k
EXTRAORDINARY
Hkkx  II — [k.M 1
PART II — Section 1
izkf/kdkj ls izdkf'kr
PUBLISHED  BY  AUTHORITY
lañ   54] ubZ fnYyh] lkseokj] fnlEcj 25] 2023@ikS"k 4] 1945 ¼'kd½
No. 54] NEW DELHI, MONDAY, DECEMBER 25, 2023/P AUSHA 4, 1945 (SAKA)
bl Hkkx esa fHkUu i`"B la[;k nh tkrh gS ftlls fd ;g vyx ladyu ds :i esa j[kk tk ldsA
Separate paging is given to this Part in order that it may be filed as a separate compilation.
xxxGIDHxxx
xxxGIDExxx
jftLVªh lañ Mhñ ,yñ—( ,u)04@0007@2003— 23 REGISTERED N O. DL—(N)0 4/000 7/200 3— 23
MINISTRY OF LA W AND JUSTICE
(Legislative Department)
New Delhi, the 25th December , 2023/Pausha 4, 1945  (Saka)
The following Act of Parliament received the assent of the President on the
25th December, 2023 and is hereby published for general information:—
THE  BHARA TIY A  NAGARIK  SURAKSHA  SANHITA,  2023
NO. 46 OF 2023
[25th December , 2023.]
An  Act  to consolidate and amend the law relating to Criminal Procedure.' metadata

In [ ]:
import re
from langchain_core.documents import Document

def extract_sections(full_text, law_name):
    
    chapter_pattern = re.compile(
        r"CHAPTER\s+([IVXLC]+)\s*\n([A-Z][A-Z\s,&\-()]+)(.*?)(?=CHAPTER\s+[IVXLC]+\s*\n|\Z)",
        re.DOTALL
    )

    documents = []

    for chapter_match in chapter_pattern.finditer(full_text):

        chapter_no = chapter_match.group(1).strip()
        chapter_name = chapter_match.group(2).strip()
        chapter_text = chapter_match.group(3).strip()

        section_pattern = re.compile(
            r"(^\d+\..*?)(?=^\d+\.|\Z)",
            re.MULTILINE | re.DOTALL
        )
        for section_match in section_pattern.finditer(chapter_text):

            section_text = section_match.group(1).strip()

            sec_no_match = re.match(
                r"(\d+)\.",
                section_text
            )

            if not sec_no_match:
                continue

            section_no = int(sec_no_match.group(1))

            documents.append(
                Document(
                    page_content=section_text,
                    metadata={
                        "law": law_name,
                        "chapter_no": chapter_no,
                        "chapter_name": chapter_name,
                        "section_no": section_no
                    }
                )
            )

    return documents



from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

bnss_path = "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023.pdf"
bsa_path= "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA SAKSHYA ADHINIYAM, 2023.pdf"
bns_path = "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA NYAYA SANHITA, 2023.pdf"

loader = PyPDFLoader(bnss_path)
documents = loader.load()

import re

full_text = "\n".join(
    doc.page_content
    for doc in documents
)

section_docs= extract_sections(full_text, "BNSS")


print(section_docs[-1].metadata)
print(len(section_docs))
# print(section_docs[-1].page_content)


In [ ]:
class Section:
    def __init__(
        self,
        law,
        chapter_no,
        chapter_name,
        section_no,
        text
    ):
        self.law = law
        self.chapter_no = chapter_no
        self.chapter_name = chapter_name
        self.section_no = section_no
        self.text = text
        # self.creationdate= date.know()
        
        
    def show(self):
        print( self.law )
        print(self.chapter_no)
        print(self.chapter_name)
        print( self.section_no )
        print( self.text )

In [ ]:
doc = section_docs[10]

s1 = Section(
    law=doc.metadata["law"],
    chapter_no=doc.metadata["chapter_no"],
    chapter_name=doc.metadata["chapter_name"],
    section_no=doc.metadata["section_no"],
    
    text=doc.page_content
)

In [ ]:
s1.show()


In [ ]:
i=0
for section in section_docs:
    print(f"{i}: {section.metadata}")
    i+=1



In [ ]:
print(section_docs[-1])
print()
print()

# print(section_docs[341])


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

bnss_path = "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023.pdf"
bsa_path= "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA SAKSHYA ADHINIYAM, 2023.pdf"
bns_path = "/Users/ravi/Desktop/Judge /NationalDocs/THE BHARATIYA NYAYA SANHITA, 2023.pdf"

loader = PyPDFLoader(bnss_path)
documents = loader.load()

import re
print(documents[0].metadata)

full_text = "\n".join(
    doc.page_content
    for doc in documents
)

In [ ]:
import re

full_text = "\n".join(
    doc.page_content
    for doc in documents
)

chapter_pattern = r"CHAPTER\s+([IVXLC]+)\s*\n([A-Z][A-Z\s,&\-()]+)"

matches = re.finditer(chapter_pattern, full_text)

i=1
for match in matches:
    print( i )
    i+=1
    print("Chapter No:", match.group(1))
    print("Chapter Name:", match.group(2))
    print()